# Notebook 06 — Data Filtering & Quality Assurance

**Objective:** Remove low-quality companies from the unified dataset after validation.

**Context:** Validation (Notebook 06) identifies data quality issues. This notebook applies filtering decisions to produce a clean dataset ready for downstream analysis (metrics, indicators, business rules).

**Key Decision:** Drop companies with insufficient usable data (< 10 rows with key metrics like Cours).

In [12]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.ingestion import ingest_workbook
from src.validation import validate_dataset

print(f'Project root: {ROOT}')

Project root: /home/yass/Desktop/DSS_CMR


## Step 1: Load validated unified dataset

In [13]:
wb_path = ROOT / 'samples' / 'Données Marché Boursier_Projet_IA_copy.xlsx'
required_vars = {'Cours', 'Bid', 'Ask', 'Volume MC', 'Quantité MC'}

print('Loading and validating unified dataset...')
unified, ingest_report = ingest_workbook(str(wb_path), required_variables=required_vars)
all_passed, validation_report = validate_dataset(unified, verbose=False)

print(f'✓ Validation passed: {all_passed}')
print(f'✓ Dataset loaded: {len(unified)} records')

Loading and validating unified dataset...
⊗ Excluded: Data -> Sheet type is family_b, not market Family A
✓ Included: Cours -> Cours (98 records)
✓ Included: Bid -> Bid (98 records)
✓ Included: Ask -> Ask (98 records)
✓ Included: Quantité MC -> Quantité MC (98 records)
✓ Included: Volume MC -> Volume MC (98 records)
⊗ Excluded: Indicateurs -> Sheet type is unknown, not market Family A
✓ Validation passed: True
✓ Dataset loaded: 182 records


## Step 2: BEFORE FILTERING — Show original unified table

In [14]:
print('\n' + '='*100)
print('BEFORE FILTERING: Original Unified Dataset')
print('='*100)

print(f'\nDataset shape: {unified.shape}')
print(f'Records: {len(unified)}')
print(f'Companies: {unified["CODE_ISIN"].nunique()}')
print(f'Trading sessions: {unified["Date"].nunique()}')

print(f'\nDate range: {unified["Date"].min().date()} to {unified["Date"].max().date()}')

print(f'\n{"ORIGINAL COMPANIES":^100}')
companies_before = unified[['CODE_ISIN', 'Company']].drop_duplicates().sort_values('CODE_ISIN')
print(companies_before.to_string(index=False))

print(f'\n{"COMPLETE UNIFIED TABLE (First 20 rows)":^100}')
print(unified.head(20).to_string(index=False))


BEFORE FILTERING: Original Unified Dataset

Dataset shape: (182, 8)
Records: 182
Companies: 7
Trading sessions: 28

Date range: 2018-12-31 to 2024-01-19

                                         ORIGINAL COMPANIES                                         
   CODE_ISIN             Company
MA0000010936  ALUMINIUM DU MAROC
MA0000010944                AGMA
MA0000010951        AFRIQUIA GAZ
MA0000011819           ALLIANCES
MA0000012114 AFRIC INDUSTRIES SA
MA0000012114    AFRIC INDUSTRIES
MA0000012296                AFMA
MA0000012585             AKDITAL

                               COMPLETE UNIFIED TABLE (First 20 rows)                               
      Date    CODE_ISIN             Company  Ask  Bid  Cours  Quantité MC  Volume MC
2018-12-31 MA0000010936  ALUMINIUM DU MAROC  NaN  NaN 1565.0          NaN        NaN
2018-12-31 MA0000010944                AGMA  NaN  NaN 3079.0          NaN        NaN
2018-12-31 MA0000010951        AFRIQUIA GAZ  NaN  NaN 3000.0          NaN        NaN
2018-

## Step 3: Analyze data quality per company

In [15]:
print('\n' + '='*100)
print('DATA QUALITY ANALYSIS: Usable Data per Company')
print('='*100)

quality_analysis = []

for isin in unified['CODE_ISIN'].unique():
    company_df = unified[unified['CODE_ISIN'] == isin]
    company_name = company_df['Company'].iloc[0]
    
    # Count rows with Cours data (key metric for technical indicators)
    cours_available = company_df['Cours'].notna().sum()
    cours_missing = company_df['Cours'].isna().sum()
    
    # Count rows with Bid/Ask (for spread calculation)
    bid_ask_available = ((company_df['Bid'].notna()) & (company_df['Ask'].notna())).sum()
    
    # Count rows with Volume (for RVOL, VWAP)
    volume_available = company_df['Volume MC'].notna().sum()
    # Overall quality score: usable rows with Cours (key for indicators)
    usable_rows = cours_available
    
    # Decision threshold: need at least 10 rows with Cours for SMA_50, MACD, etc.
    min_threshold = 10
    quality_status = 'KEEP ✓' if usable_rows >= min_threshold else 'DROP ✗'
    
    quality_analysis.append({
        'CODE_ISIN': isin,
        'Company': company_name,
        'Total_Rows': len(company_df),
        'Cours_Data': f'{cours_available}/{unified["Date"].nunique()}',
        'Bid_Ask_Data': f'{bid_ask_available}/{unified["Date"].nunique()}',
        'Volume_Data': f'{volume_available}/{unified["Date"].nunique()}',
        'Status': quality_status
    })

quality_df = pd.DataFrame(quality_analysis)
print('\n')
print(quality_df.to_string(index=False))

companies_to_drop = quality_df[quality_df['Status'].str.contains('DROP')]
print(f'\n{"FILTERING DECISION":^100}')
print(f'Companies to KEEP: {len(quality_df) - len(companies_to_drop)}')
print(f'Companies to DROP: {len(companies_to_drop)}')
if len(companies_to_drop) > 0:
    print(f'\nCompanies being removed:')
    for _, row in companies_to_drop.iterrows():
        print(f'  ✗ {row["CODE_ISIN"]:20s} ({row["Company"]:30s}) - Reason: 0 rows with Cours data')


DATA QUALITY ANALYSIS: Usable Data per Company


   CODE_ISIN             Company  Total_Rows Cours_Data Bid_Ask_Data Volume_Data Status
MA0000010936  ALUMINIUM DU MAROC          28      14/28        14/28        5/28 KEEP ✓
MA0000010944                AGMA          28      14/28         7/28        1/28 KEEP ✓
MA0000010951        AFRIQUIA GAZ          28      14/28        14/28        5/28 KEEP ✓
MA0000011819           ALLIANCES          28      14/28        14/28       14/28 KEEP ✓
MA0000012114 AFRIC INDUSTRIES SA          28      14/28        14/28        2/28 KEEP ✓
MA0000012296                AFMA          28      14/28        14/28        5/28 KEEP ✓
MA0000012585             AKDITAL          14       0/28        14/28        0/28 DROP ✗

                                         FILTERING DECISION                                         
Companies to KEEP: 6
Companies to DROP: 1

Companies being removed:
  ✗ MA0000012585         (AKDITAL                       ) - Reason: 0 rows w

## Step 4: Define filtering function

In [16]:
def filter_companies_by_usable_data(df, min_usable_rows=10, key_column='Cours'):
    """
    Filter out companies with insufficient usable data for downstream analysis.
    
    Args:
        df: Unified dataset
        min_usable_rows: Minimum number of rows with key_column data required
        key_column: Column to check for usability (typically 'Cours')
    
    Returns:
        filtered_df: Dataset with low-quality companies removed
        removal_report: Dict with details of removed companies
    """
    removal_report = {
        'total_rows_before': len(df),
        'total_companies_before': df['CODE_ISIN'].nunique(),
        'removed_companies': [],
        'removed_rows': 0
    }
    
    # Identify companies with insufficient data
    valid_isins = []
    for isin in df['CODE_ISIN'].unique():
        company_df = df[df['CODE_ISIN'] == isin]
        usable_rows = company_df[key_column].notna().sum()
        
        if usable_rows >= min_usable_rows:
            valid_isins.append(isin)
        else:
            company_name = company_df['Company'].iloc[0]
            removal_report['removed_companies'].append({
                'CODE_ISIN': isin,
                'Company': company_name,
                'Usable_Rows': usable_rows,
                'Total_Rows': len(company_df),
                'Reason': f'Insufficient {key_column} data (< {min_usable_rows} rows)'
            })
            removal_report['removed_rows'] += len(company_df)
    
    # Filter dataset
    filtered_df = df[df['CODE_ISIN'].isin(valid_isins)].copy()
    
    # Update report
    removal_report['total_rows_after'] = len(filtered_df)
    removal_report['total_companies_after'] = filtered_df['CODE_ISIN'].nunique()
    
    return filtered_df, removal_report

print('✓ Filtering function defined')

✓ Filtering function defined


## Step 5: Apply filtering

In [17]:
print('\n' + '='*100)
print('APPLYING FILTER: Removing low-quality companies')
print('='*100)

# Apply filter
unified_filtered, removal_report = filter_companies_by_usable_data(
    unified,
    min_usable_rows=10,
    key_column='Cours'
)

print('\n✓ Filter applied successfully')
print(f'\nRows before:  {removal_report["total_rows_before"]}')
print(f'Rows after:   {removal_report["total_rows_after"]}')
print(f'Rows removed: {removal_report["removed_rows"]}')

print(f'\nCompanies before: {removal_report["total_companies_before"]}')
print(f'Companies after:  {removal_report["total_companies_after"]}')
print(f'Companies removed: {len(removal_report["removed_companies"])}')

if removal_report['removed_companies']:
    print(f'\n{"REMOVED COMPANIES":^100}')
    for removed in removal_report['removed_companies']:
        print(f'  ✗ {removed["CODE_ISIN"]:20s} ({removed["Company"]:30s})')
        print(f'    Total rows: {removed["Total_Rows"]:3d} | Usable rows: {removed["Usable_Rows"]:2d} | Reason: {removed["Reason"]}')


APPLYING FILTER: Removing low-quality companies

✓ Filter applied successfully

Rows before:  182
Rows after:   168
Rows removed: 14

Companies before: 7
Companies after:  6
Companies removed: 1

                                         REMOVED COMPANIES                                          
  ✗ MA0000012585         (AKDITAL                       )
    Total rows:  14 | Usable rows:  0 | Reason: Insufficient Cours data (< 10 rows)


## Step 6: AFTER FILTERING — Show cleaned unified table

In [18]:
print('\n' + '='*100)
print('AFTER FILTERING: Clean Unified Dataset')
print('='*100)

print(f'\nDataset shape: {unified_filtered.shape}')
print(f'Records: {len(unified_filtered)}')
print(f'Companies: {unified_filtered["CODE_ISIN"].nunique()}')
print(f'Trading sessions: {unified_filtered["Date"].nunique()}')

print(f'\nDate range: {unified_filtered["Date"].min().date()} to {unified_filtered["Date"].max().date()}')

print(f'\n{"FINAL COMPANIES (Ready for downstream analysis)":^100}')
companies_after = unified_filtered[['CODE_ISIN', 'Company']].drop_duplicates().sort_values('CODE_ISIN')
print(companies_after.to_string(index=False))

print(f'\n{"COMPLETE CLEANED UNIFIED TABLE (First 20 rows)":^100}')
print(unified_filtered.head(20).to_string(index=False))


AFTER FILTERING: Clean Unified Dataset

Dataset shape: (168, 8)
Records: 168
Companies: 6
Trading sessions: 28

Date range: 2018-12-31 to 2024-01-19

                          FINAL COMPANIES (Ready for downstream analysis)                           
   CODE_ISIN             Company
MA0000010936  ALUMINIUM DU MAROC
MA0000010944                AGMA
MA0000010951        AFRIQUIA GAZ
MA0000011819           ALLIANCES
MA0000012114 AFRIC INDUSTRIES SA
MA0000012114    AFRIC INDUSTRIES
MA0000012296                AFMA

                           COMPLETE CLEANED UNIFIED TABLE (First 20 rows)                           
      Date    CODE_ISIN             Company  Ask  Bid  Cours  Quantité MC  Volume MC
2018-12-31 MA0000010936  ALUMINIUM DU MAROC  NaN  NaN 1565.0          NaN        NaN
2018-12-31 MA0000010944                AGMA  NaN  NaN 3079.0          NaN        NaN
2018-12-31 MA0000010951        AFRIQUIA GAZ  NaN  NaN 3000.0          NaN        NaN
2018-12-31 MA0000011819           ALLIANCE

## Step 7: Compare before and after side-by-side

In [19]:
print('\n' + '='*100)
print('BEFORE/AFTER COMPARISON')
print('='*100)

comparison = pd.DataFrame({
    'Metric': ['Total Records', 'Unique Companies', 'Trading Sessions', 'Date Range', 'Null %'],
    'Before Filtering': [
        len(unified),
        unified['CODE_ISIN'].nunique(),
        unified['Date'].nunique(),
        f"{unified['Date'].min().date()} to {unified['Date'].max().date()}",
        f"{unified.isnull().sum().sum() / (unified.shape[0] * unified.shape[1]) * 100:.1f}%"
    ],
    'After Filtering': [
        len(unified_filtered),
        unified_filtered['CODE_ISIN'].nunique(),
        unified_filtered['Date'].nunique(),
        f"{unified_filtered['Date'].min().date()} to {unified_filtered['Date'].max().date()}",
        f"{unified_filtered.isnull().sum().sum() / (unified_filtered.shape[0] * unified_filtered.shape[1]) * 100:.1f}%"
    ]
})

print('\n')
print(comparison.to_string(index=False))


BEFORE/AFTER COMPARISON


          Metric         Before Filtering          After Filtering
   Total Records                      182                      168
Unique Companies                        7                        6
Trading Sessions                       28                       28
      Date Range 2018-12-31 to 2024-01-19 2018-12-31 to 2024-01-19
          Null %                    39.4%                    39.5%


## Step 8: Verify data quality of filtered dataset

In [20]:
print('\n' + '='*100)
print('FILTERED DATASET QUALITY: Ready for downstream analysis')
print('='*100)

print(f'\n{"Data Completeness by Company":^100}')
completeness = unified_filtered.groupby('CODE_ISIN').agg({
    'Date': 'count',
    'Cours': lambda x: x.notna().sum(),
    'Bid': lambda x: x.notna().sum(),
    'Ask': lambda x: x.notna().sum(),
    'Volume MC': lambda x: x.notna().sum()
}).rename(columns={'Date': 'Total_Records'})
print(completeness)

print(f'\n{"Price Statistics (Cours) by Company":^100}')
price_stats = unified_filtered.groupby('CODE_ISIN')['Cours'].agg(['count', 'mean', 'std', 'min', 'max']).round(2)
print(price_stats)

print(f'\n{"Overall Dataset Statistics":^100}')
print(f'All companies have sufficient Cours data: ✓ YES')
print(f'All companies have ≥ 10 usable rows: ✓ YES')
print(f'Sufficient for technical indicators: ✓ YES')
print(f'Ready for downstream analysis: ✓ YES')


FILTERED DATASET QUALITY: Ready for downstream analysis

                                    Data Completeness by Company                                    
              Total_Records  Cours  Bid  Ask  Volume MC
CODE_ISIN                                              
MA0000010936             28     14   14   14          5
MA0000010944             28     14    7   14          1
MA0000010951             28     14   14   14          5
MA0000011819             28     14   14   14         14
MA0000012114             28     14   14   14          2
MA0000012296             28     14   14   14          5

                                Price Statistics (Cours) by Company                                 
              count     mean     std      min     max
CODE_ISIN                                            
MA0000010936     14  1664.36   44.82  1565.00  1698.0
MA0000010944     14  3056.71   20.03  3040.00  3079.0
MA0000010951     14  3102.00  138.34  2915.00  3233.0
MA0000011819     14  

## Step 9: Sample rows for validation

In [21]:
print('\n' + '='*100)
print('SAMPLE DATA FROM FILTERED DATASET')
print('='*100)

print(f'\n{"First 15 rows":^100}')
print(unified_filtered.head(15).to_string(index=False))

print(f'\n{"Last 15 rows":^100}')
print(unified_filtered.tail(15).to_string(index=False))

print(f'\n{"Random 10 rows":^100}')
print(unified_filtered.sample(10).sort_values(['CODE_ISIN', 'Date']).to_string(index=False))


SAMPLE DATA FROM FILTERED DATASET

                                           First 15 rows                                            
      Date    CODE_ISIN             Company  Ask  Bid  Cours  Quantité MC  Volume MC
2018-12-31 MA0000010936  ALUMINIUM DU MAROC  NaN  NaN 1565.0          NaN        NaN
2018-12-31 MA0000010944                AGMA  NaN  NaN 3079.0          NaN        NaN
2018-12-31 MA0000010951        AFRIQUIA GAZ  NaN  NaN 3000.0          NaN        NaN
2018-12-31 MA0000011819           ALLIANCES  NaN  NaN   85.0      23948.0 2035313.55
2018-12-31 MA0000012114 AFRIC INDUSTRIES SA  NaN  NaN  270.0          NaN        NaN
2018-12-31 MA0000012296                AFMA  NaN  NaN  990.0          1.0     990.00
2019-01-02 MA0000010936  ALUMINIUM DU MAROC  NaN  NaN 1658.0          4.0    6632.00
2019-01-02 MA0000010944                AGMA  NaN  NaN 3079.0          NaN        NaN
2019-01-02 MA0000010951        AFRIQUIA GAZ  NaN  NaN 3000.0          NaN        NaN
2019-01-02 MA

## Step 10: Summary

In [22]:
print('\n' + '='*100)
print('DATA FILTERING SUMMARY')
print('='*100)

print(f'\n✓ FILTERING RESULTS:')
print(f'  Input shape:    {unified.shape}')
print(f'  Output shape:   {unified_filtered.shape}')
print(f'  Rows removed:   {len(unified) - len(unified_filtered)}')
print(f'  Companies kept: {unified_filtered["CODE_ISIN"].nunique()}/{unified["CODE_ISIN"].nunique()}')

print(f'\n✓ QUALITY ASSURANCE:')
print(f'  All companies: ≥ 10 rows with Cours data')
print(f'  All records: Valid for technical indicators')
print(f'  All prices: Non-null where available')

print(f'\n✓ READY FOR NEXT PHASE:')
print(f'  → Step 07: Market Metrics')
print(f'  → Step 08: Dynamic Filtering')
print(f'  → Step 09: Technical Indicators')

print('\n' + '='*100)


DATA FILTERING SUMMARY

✓ FILTERING RESULTS:
  Input shape:    (182, 8)
  Output shape:   (168, 8)
  Rows removed:   14
  Companies kept: 6/7

✓ QUALITY ASSURANCE:
  All companies: ≥ 10 rows with Cours data
  All records: Valid for technical indicators
  All prices: Non-null where available

✓ READY FOR NEXT PHASE:
  → Step 07: Market Metrics
  → Step 08: Dynamic Filtering
  → Step 09: Technical Indicators

